# 06 • Cruzar tabelas e mudar o formato

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [1]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [2]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
Dimensões: (344, 8)
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Series
   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen
                   ausentes  percentual
species                   0    0.000000
island                    0    0.000000
bill_length_mm            2    0.581395
bill_depth_mm             

### Como pensar antes de um merge

Imagine uma tabela de medições com várias linhas Adelie e um pequeno cadastro com uma única tradução para Adelie. Cada observação deve encontrar exatamente uma tradução. Essa relação é many_to_one: muitas observações para um cadastro. Em `p.merge(nomes, on='species', how='left', validate='many_to_one')`, nomes é a tabela da direita, on indica a coluna comum e left preserva as observações da esquerda. Se o cadastro repetir Adelie, cada observação dessa espécie poderá aparecer duas vezes. A validação interrompe a operação para você investigar.

Confira três coisas depois do cruzamento: quantidade de linhas, valores sem correspondência e somas das medidas que deveriam permanecer iguais. Apenas conferir o tamanho não detecta todos os problemas de correspondência. indicator=True cria a coluna _merge; both indica correspondência, left_only indica que não houve par à direita.

Ao reorganizar dados, pergunte se quer mudar somente o formato ou também resumir. A pivot_table de massa por espécie e sexo calcula médias: ela agrega várias observações em uma célula. Já melt não calcula médias; transforma duas colunas de medidas em uma coluna de nomes de medidas e outra de valores. Se você derrete duas medidas de 344 animais, o resultado terá 688 linhas. Essas linhas agora representam animal–medida, e não animais distintos.

### Aprofundamento e cuidados

`merge` cruza tabelas por chaves. Um left join preserva todas as linhas da esquerda, mas pode multiplicá-las se a chave da direita se repetir. Defina a granularidade de cada lado e use `validate="many_to_one"` quando várias linhas da esquerda devem encontrar uma única linha na direita. `indicator=True` ajuda a localizar chaves sem correspondência. No pandas, chaves ausentes podem corresponder entre si; valide-as antes se isso não fizer sentido no seu processo.

`concat` empilha tabelas; `ignore_index=True` cria novos rótulos de linha. Ele não resolve duplicatas ou diferenças de significado. `pivot` reorganiza e exige combinações únicas; `pivot_table` permite agregar duplicatas. `melt` transforma colunas de medidas em linhas, formando uma tabela longa.

A tabela auxiliar de nomes de espécies usada aqui é DIDÁTICA. As medições de pinguins continuam sendo reais. Não apresente a tradução construída como um novo levantamento científico.


## Exemplo guiado

In [3]:
nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

_merge
both          344
left_only       0
right_only      0
Name: count, dtype: int64


## Exercício 06.1
Cruze as traduções e confirme que a quantidade de linhas permaneceu igual.

In [4]:
q = p.merge(nomes, on='species', how='left', validate='many_to_one')
assert len(q)==len(p)
print(q[['species','nome_pt']].head())

  species nome_pt
0  Adelie  Adélia
1  Adelie  Adélia
2  Adelie  Adélia
3  Adelie  Adélia
4  Adelie  Adélia


## Exercício 06.2
Construa uma tabela de massa média por espécie (linhas) e sexo (colunas).

In [5]:
print(p.pivot_table(index='species', columns='sex', values='body_mass_g', aggfunc='mean', observed=True))

sex             female         male
species                            
Adelie     3368.835616  4043.493151
Chinstrap  3527.205882  3938.970588
Gentoo     4679.741379  5484.836066


## Exercício 06.3
Transforme bill_length_mm e bill_depth_mm em linhas, preservando species como identificador.

In [6]:
print(p.melt(id_vars=['species'], value_vars=['bill_length_mm','bill_depth_mm'], var_name='medida', value_name='mm').head())

  species          medida    mm
0  Adelie  bill_length_mm  39.1
1  Adelie  bill_length_mm  39.5
2  Adelie  bill_length_mm  40.3
3  Adelie  bill_length_mm   NaN
4  Adelie  bill_length_mm  36.7


## Exercício 06.4
Divida p em dois blocos e una novamente. Confirme que os dados são iguais.

In [7]:
q = pd.concat([p.iloc[:100],p.iloc[100:]], ignore_index=True)
pd.testing.assert_frame_equal(p,q)
print('Reconstrução correta')

Reconstrução correta


## Exercício 06.5
Duplique uma chave da tabela de tradução e demonstre que validate impede a multiplicação silenciosa.

In [8]:
ruim = pd.concat([nomes,nomes.iloc[:1]], ignore_index=True)
try:
    p.merge(ruim, on='species', validate='many_to_one')
except pd.errors.MergeError as erro:
    print(type(erro).__name__, str(erro))

MergeError Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 species
 Adelie ...
